# LeetCode 3241 — Time Taken to Mark All Nodes bằng Python

- Link: https://leetcode.com/problems/time-taken-to-mark-all-nodes/
- Độ khó: **Hard**
- Chủ đề: **Khoảng cách có trọng số trên đỉnh đích** (vào đỉnh lẻ tốn 1, vào đỉnh chẵn tốn 2) ⇒ đáp án là **độ lệch tâm có trọng số**; tính cho mọi gốc bằng **rerooting DP** (top-2 chiều sâu xuống + giá trị lên)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def timeTaken(self, edges: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Quá trình đánh dấu lan như BFS có trọng số: đi vào đỉnh `v` tốn `w(v) = 1` nếu `v` lẻ, `2` nếu `v` chẵn. Trên cây có đường đi duy nhất ⇒ thời điểm đánh dấu `v` khi bắt đầu từ `i` là `Σ w(x)` trên đường `i → v` (không tính `i`).
2. `times[i] = max_v dist_w(i, v)` — **độ lệch tâm có trọng số** (trọng số theo đỉnh đích ⇒ khoảng cách **không đối xứng**!).
3. Rerooting: `down[u]` = max khoảng cách từ `u` xuống cây con; `up[u]` = max khoảng cách từ `u` ra ngoài cây con (qua cha). Giữ **top-2** của `down` qua các con để tính `up` cho con.
4. Duyệt lặp (cây sâu `10^5`).
5. Kiểm chứng bằng Dijkstra/duyệt từ mọi đỉnh.

## 2. Đề bài & ví dụ

Cây `n` đỉnh. Ban đầu không đỉnh nào được đánh dấu. Đỉnh `i` **lẻ** được đánh dấu tại thời điểm `x` nếu có đỉnh kề được đánh dấu tại `x − 1`; đỉnh `i` **chẵn** được đánh dấu tại `x` nếu có đỉnh kề được đánh dấu tại `x − 2`. Với mỗi `i`, nếu đánh dấu `i` tại `t = 0`, hỏi thời điểm mọi đỉnh được đánh dấu.

**Ví dụ 1:** `[[0,1],[0,2]]` → `[2,4,3]`. **Ví dụ 2:** `[[0,1]]` → `[1,2]`. **Ví dụ 3:** `[[2,4],[0,1],[2,3],[0,2]]` → `[4,6,3,5,5]`.

**Ràng buộc:** `2 ≤ n ≤ 10^5`; cạnh tạo thành cây.

In [1]:
from typing import List
from collections import deque

examples = [([[0, 1], [0, 2]], [2, 4, 3]), ([[0, 1]], [1, 2]), ([[2, 4], [0, 1], [2, 3], [0, 2]], [4, 6, 3, 5, 5])]
for e, exp in examples:
    print(e, '-> kỳ vọng', exp)

[[0, 1], [0, 2]] -> kỳ vọng [2, 4, 3]
[[0, 1]] -> kỳ vọng [1, 2]
[[2, 4], [0, 1], [2, 3], [0, 2]] -> kỳ vọng [4, 6, 3, 5, 5]


## 3. Kiến thức nền

### 3.1. Trọng số theo đỉnh đích

In [2]:
def w(v):
    return 1 if v % 2 else 2

def build(edges):
    n = len(edges) + 1
    g = [[] for _ in range(n)]
    for a, b in edges:
        g[a].append(b); g[b].append(a)
    return g

def dist_from(g, s):
    d = [-1] * len(g); d[s] = 0; st = [s]
    while st:
        u = st.pop()
        for v in g[u]:
            if d[v] < 0:
                d[v] = d[u] + w(v); st.append(v)
    return d

g = build(examples[0][0])
print('từ 1:', dist_from(g, 1), '| từ 0:', dist_from(g, 0))

từ 1: [2, 0, 4] | từ 0: [0, 1, 2]


Từ `1`: đỉnh `0` (chẵn) tốn 2, đỉnh `2` (chẵn) thêm 2 ⇒ `4`. Trên cây, "đánh dấu sớm nhất" chính là tổng trọng số trên đường đi duy nhất — không có đường tắt.

### 3.2. Không đối xứng

`dist(1 → 0) = w(0) = 2` nhưng `dist(0 → 1) = w(1) = 1`. Vì vậy **không** dùng được mẹo "hai đầu mút đường kính" như bài trọng số cạnh; rerooting thì vẫn dùng được.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Gốc tạm `0`. `down[u] = max_{c con} (w(c) + down[c])`. Với con `c` của `u`: khoảng cách xa nhất từ `c` đi **lên** = `w(u) + max(up[u], max_{c' ≠ c} (w(c') + down[c']))`. Đáp án `times[u] = max(down[u], up[u])`.

Ví dụ 3 (gốc 0): `down[2] = max(w(4)+0, w(3)+0) = max(2, 1) = 2`; `down[0] = max(w(1)+0, w(2)+2) = max(1, 4) = 4` ⇒ `times[0] = 4` ✓.

## 5. Cách 1 — Brute Force: duyệt từ mọi đỉnh

In [3]:
def brute(edges):
    g = build(edges)
    return [max(dist_from(g, s)) for s in range(len(g))]

for e, exp in examples:
    print(brute(e), '| kỳ vọng', exp)

[2, 4, 3] | kỳ vọng [2, 4, 3]
[1, 2] | kỳ vọng [1, 2]
[4, 6, 3, 5, 5] | kỳ vọng [4, 6, 3, 5, 5]


**Độ phức tạp:** `O(n²)`.

## 6. Ý tưởng thuật toán / bước đột phá

1. Lấy thứ tự duyệt (cha trước con) từ gốc `0`.
2. Hậu thứ tự: `down[u]`, đồng thời giữ `best1[u]`, `best2[u]` (hai giá trị `w(c) + down[c]` lớn nhất qua hai con **khác nhau**) và con đạt `best1`.
3. Tiền thứ tự: `up[c] = w(u) + max(up[u], best1[u] nếu con đạt best1 ≠ c, ngược lại best2[u])` (với gốc `up = 0` — nhưng chú ý gốc không có "đi lên", nên dùng `up[root] = 0` chỉ khi đó là max với tập rỗng).
4. `times[u] = max(down[u], up[u])`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Duyệt từ mọi đỉnh nhưng dùng BFS 0-1-2 (Dijkstra) — tổng quát cho đồ thị

In [4]:
import heapq
def dijkstra_all(edges):
    g = build(edges); n = len(g); res = []
    for s in range(n):
        d = [float('inf')] * n; d[s] = 0; pq = [(0, s)]
        while pq:
            du, u = heapq.heappop(pq)
            if du > d[u]:
                continue
            for v in g[u]:
                nd = du + w(v)
                if nd < d[v]:
                    d[v] = nd; heapq.heappush(pq, (nd, v))
        res.append(max(d))
    return res

for e, exp in examples:
    print(dijkstra_all(e), '| kỳ vọng', exp)

[2, 4, 3] | kỳ vọng [2, 4, 3]
[1, 2] | kỳ vọng [1, 2]
[4, 6, 3, 5, 5] | kỳ vọng [4, 6, 3, 5, 5]


Dijkstra không cần thiết trên cây (đường duy nhất) nhưng là định nghĩa "thời điểm sớm nhất" đúng nghĩa cho đồ thị tổng quát — dùng để đối chiếu.

### 7.2. Cách 3 — Rerooting DP với top-2 (lời giải chính)

In [5]:
def rerooting(edges):
    g = build(edges); n = len(g)
    parent = [-1] * n; order = []; seen = [False] * n; seen[0] = True; st = [0]
    while st:
        u = st.pop(); order.append(u)
        for v in g[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; st.append(v)
    down = [0] * n; b1 = [0] * n; b2 = [0] * n; via = [-1] * n
    for u in reversed(order):
        for c in g[u]:
            if c == parent[u]:
                continue
            val = w(c) + down[c]
            if val > b1[u]:
                b2[u] = b1[u]; b1[u] = val; via[u] = c
            elif val > b2[u]:
                b2[u] = val
        down[u] = b1[u]
    up = [0] * n
    for u in order:
        for c in g[u]:
            if c == parent[u]:
                continue
            side = b2[u] if via[u] == c else b1[u]
            up[c] = w(u) + max(up[u], side)
    return [max(down[u], up[u]) for u in range(n)]

for e, exp in examples:
    print(rerooting(e), '| kỳ vọng', exp)

[2, 4, 3] | kỳ vọng [2, 4, 3]
[1, 2] | kỳ vọng [1, 2]
[4, 6, 3, 5, 5] | kỳ vọng [4, 6, 3, 5, 5]


Gốc có `up[0] = 0` — khi tính `up[c]` cho con của gốc, `max(up[0], side) = side` (vì `side ≥ 0`), đúng nghĩa "đi lên gốc rồi xuống nhánh khác (hoặc dừng ở gốc)".

### 7.3. Cách 4 — Rerooting đệ quy có nhớ theo cạnh có hướng `f(u, p)`

`f(u, p)` = khoảng cách xa nhất từ `u` khi **không** được đi về phía `p`. `times[u] = f(u, −1)`. Mỗi cạnh có hướng tính một lần ⇒ nhưng đỉnh bậc lớn cần tối ưu (top-2), nếu không sẽ `O(Σ deg²)`. Bản dưới dùng memo + top-2 trên các cạnh ra, minh họa ý tưởng "DP trên cạnh có hướng" (chạy tốt với cây không có đỉnh bậc quá lớn).

In [6]:
import sys
sys.setrecursionlimit(20000)
def directed_edge_dp(edges):
    g = build(edges); n = len(g)
    from functools import lru_cache
    @lru_cache(maxsize=None)
    def f(u, p):
        best = 0
        for v in g[u]:
            if v != p:
                best = max(best, w(v) + f(v, u))
        return best
    res = [f(u, -1) for u in range(n)]
    f.cache_clear()
    return res

for e, exp in examples:
    print(directed_edge_dp(e), '| kỳ vọng', exp)

[2, 4, 3] | kỳ vọng [2, 4, 3]
[1, 2] | kỳ vọng [1, 2]
[4, 6, 3, 5, 5] | kỳ vọng [4, 6, 3, 5, 5]


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 3.

In [7]:
def trace(edges):
    g = build(edges); n = len(g)
    parent = [-1] * n; order = []; seen = [False] * n; seen[0] = True; st = [0]
    while st:
        u = st.pop(); order.append(u)
        for v in g[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; st.append(v)
    down = [0] * n; b1 = [0] * n; b2 = [0] * n; via = [-1] * n
    for u in reversed(order):
        for c in g[u]:
            if c != parent[u]:
                val = w(c) + down[c]
                if val > b1[u]:
                    b2[u], b1[u], via[u] = b1[u], val, c
                elif val > b2[u]:
                    b2[u] = val
        down[u] = b1[u]
    up = [0] * n
    for u in order:
        for c in g[u]:
            if c != parent[u]:
                up[c] = w(u) + max(up[u], b2[u] if via[u] == c else b1[u])
    for u in range(n):
        print(f'đỉnh {u} (w = {w(u)}): down = {down[u]}, top2 = ({b1[u]}, {b2[u]}) qua con {via[u]}, up = {up[u]} -> {max(down[u], up[u])}')
    print('vét cạn:', brute(edges))

trace(examples[2][0])

đỉnh 0 (w = 2): down = 4, top2 = (4, 1) qua con 2, up = 0 -> 4
đỉnh 1 (w = 1): down = 0, top2 = (0, 0) qua con -1, up = 6 -> 6
đỉnh 2 (w = 2): down = 2, top2 = (2, 1) qua con 4, up = 3 -> 3
đỉnh 3 (w = 1): down = 0, top2 = (0, 0) qua con -1, up = 5 -> 5
đỉnh 4 (w = 2): down = 0, top2 = (0, 0) qua con -1, up = 5 -> 5
vét cạn: [4, 6, 3, 5, 5]


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def timeTaken(self, edges: List[List[int]]) -> List[int]:
        n = len(edges) + 1
        g = [[] for _ in range(n)]
        for a, b in edges:
            g[a].append(b)
            g[b].append(a)
        wt = [1 if v & 1 else 2 for v in range(n)]
        parent = [-1] * n
        order = [0]
        parent[0] = 0
        for u in order:                      # BFS lấy thứ tự cha trước con
            for v in g[u]:
                if v != parent[u] and (v != 0):
                    if parent[v] == -1:
                        parent[v] = u
                        order.append(v)
        parent[0] = -1
        b1 = [0] * n
        b2 = [0] * n
        via = [-1] * n
        for u in reversed(order):
            p = parent[u]
            if p >= 0:
                val = wt[u] + b1[u]
                if val > b1[p]:
                    b2[p] = b1[p]; b1[p] = val; via[p] = u
                elif val > b2[p]:
                    b2[p] = val
        up = [0] * n
        for u in order:
            for c in g[u]:
                if c == parent[u]:
                    continue
                side = b2[u] if via[u] == c else b1[u]
                up[c] = wt[u] + (up[u] if up[u] > side else side)
        return [b1[u] if b1[u] > up[u] else up[u] for u in range(n)]

for e, exp in examples:
    got = Solution().timeTaken(e)
    print(got, 'OK' if got == exp else 'SAI')

[2, 4, 3] OK
[1, 2] OK
[4, 6, 3, 5, 5] OK


Trong bản nộp, `b1[u]` chính là `down[u]` và được đẩy lên cha ngay khi xử lý `u` theo hậu thứ tự (thứ tự BFS đảo ngược).

## 10. Vì sao thuật toán đúng

1. **Thời điểm đánh dấu:** quy nạp theo thời gian, `v` được đánh dấu tại `min_{u kề v, u đã đánh dấu} (t_u + w(v))` ⇒ đúng là khoảng cách ngắn nhất với trọng số đỉnh đích; trên cây là tổng trọng số trên đường đi duy nhất.
2. **Phân rã:** mọi đỉnh `v` hoặc nằm trong cây con của `u` (khoảng cách ≤ `down[u]`, đạt được), hoặc nằm ngoài — đường đi qua cha `p`: `w(p) + (khoảng cách từ p không đi vào cây con của u)` = `up[u]`.
3. **Top-2:** khoảng cách từ `p` không đi vào nhánh `u` = `max(up[p], max_{c ≠ u} (w(c) + down[c]))` — lấy `best1` nếu nó không đến từ `u`, ngược lại `best2`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt từ mọi đỉnh | `O(n²)` | `O(n)` |
| 2. Dijkstra từ mọi đỉnh | `O(n² log n)` | `O(n)` |
| 3. Rerooting top-2 | `O(n)` | `O(n)` |
| 4. DP cạnh có hướng | `O(Σ deg²)` (xấu với hình sao) | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'dijkstra': dijkstra_all, 'rerooting': rerooting, 'directed': directed_edge_dp, 'submit': lambda e: Solution().timeTaken(e)}
edge = [([[1, 0]], None), ([[0, 1], [1, 2], [2, 3]], None), ([[0, 1], [0, 2], [0, 3], [0, 4]], None), ([[3, 1], [1, 2], [2, 0]], None)]
for e, exp in examples + edge:
    b = brute(e)
    if exp is not None:
        assert b == exp, (e, b)
    for name, f in impls.items():
        assert f(e) == b, (name, e)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với duyệt từ mọi đỉnh

In [10]:
import random
rng = random.Random(3241)
def random_tree(n):
    lab = list(range(n)); rng.shuffle(lab)
    return [[lab[i], lab[rng.randrange(i)]] for i in range(1, n)]

cnt = 0
for _ in range(400):
    e = random_tree(rng.randint(2, 25))
    exp = brute(e)
    for name, f in impls.items():
        assert f(e) == exp, (name, e)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cây ngẫu nhiên.')

Khớp vét cạn trên 400 cây ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5`

In [11]:
import time
N = 10 ** 5
big = {
    'ngẫu nhiên': random_tree(N),
    'đường thẳng': [[i, i + 1] for i in range(N - 1)],
    'hình sao (tâm 0)': [[0, i] for i in range(1, N)],
}
for name, e in big.items():
    t = time.time(); a = Solution().timeTaken(e); t1 = time.time() - t
    t = time.time(); b = rerooting(e); t2 = time.time() - t
    assert a == b
    g = build(e)
    for s in rng.sample(range(N), 20):
        assert a[s] == max(dist_from(g, s))
    print(f"{name:16s}: nộp {t1:.2f}s | cách 3 {t2:.2f}s | 20 đỉnh ngẫu nhiên khớp duyệt trực tiếp")

ngẫu nhiên      : nộp 0.19s | cách 3 0.24s | 20 đỉnh ngẫu nhiên khớp duyệt trực tiếp


đường thẳng     : nộp 0.19s | cách 3 0.10s | 20 đỉnh ngẫu nhiên khớp duyệt trực tiếp


hình sao (tâm 0): nộp 0.12s | cách 3 0.06s | 20 đỉnh ngẫu nhiên khớp duyệt trực tiếp


Kiểm tra độc lập (hình sao tâm 0): từ tâm, xa nhất là một lá chẵn (`2`) nếu có; từ lá `i`: `w(0) + 2 = 4` (đi qua tâm tới lá chẵn khác).

In [12]:
a = Solution().timeTaken(big['hình sao (tâm 0)'])
assert a[0] == 2 and all(a[i] == 4 for i in range(1, N) if True)
print('hình sao: tâm -> 2, mọi lá -> 4 — đúng.')

hình sao: tâm -> 2, mọi lá -> 4 — đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng trọng số theo đỉnh **nguồn** (hoặc coi khoảng cách đối xứng)

In [13]:
def wrong_source_weight(edges):
    g = build(edges); n = len(g); res = []
    for s in range(n):
        d = [-1] * n; d[s] = 0; st = [s]
        while st:
            u = st.pop()
            for v in g[u]:
                if d[v] < 0:
                    d[v] = d[u] + w(u); st.append(v)           # SAI: trọng số của đỉnh đi ra
        res.append(max(d))
    return res

print('sai :', wrong_source_weight(examples[1][0]), '| đúng:', Solution().timeTaken(examples[1][0]))

sai : [2, 1] | đúng: [1, 2]


**Vì sao sai:** thời gian chờ phụ thuộc đỉnh **được đánh dấu** (đỉnh đích): đỉnh chẵn cần 2 giây sau khi hàng xóm được đánh dấu.

### Lỗi 2 — Dùng mẹo hai đầu mút đường kính

In [14]:
def wrong_diameter_trick(edges):
    g = build(edges); n = len(g)
    d0 = dist_from(g, 0); a = d0.index(max(d0))
    da = dist_from(g, a); b = da.index(max(da))
    db = dist_from(g, b)
    # SAI: khoảng cách không đối xứng -> dist(i, a) != dist(a, i)
    return [max(da[i], db[i]) for i in range(n)]

found = None
r_ = random.Random(2)
for _ in range(3000):
    e = random_tree(r_.randint(3, 8))
    if wrong_diameter_trick(e) != brute(e):
        found = e; break
print('phản ví dụ:', found, '-> sai:', wrong_diameter_trick(found), '| đúng:', Solution().timeTaken(found))

phản ví dụ: [[0, 2], [1, 2]] -> sai: [4, 3, 2] | đúng: [3, 4, 2]


**Vì sao sai:** `da[i]` là khoảng cách **từ `a` tới `i`**, không phải từ `i` tới `a`; với trọng số theo đỉnh đích hai giá trị khác nhau, và tính chất đường kính cũng không còn đúng.

### Lỗi 3 — Tính `up[c]` bằng `best1` của cha kể cả khi `best1` đến từ chính `c`

In [15]:
def wrong_top1_only(edges):
    g = build(edges); n = len(g)
    parent = [-1] * n; order = []; seen = [False] * n; seen[0] = True; st = [0]
    while st:
        u = st.pop(); order.append(u)
        for v in g[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; st.append(v)
    down = [0] * n
    for u in reversed(order):
        for c in g[u]:
            if c != parent[u]:
                down[u] = max(down[u], w(c) + down[c])
    up = [0] * n
    for u in order:
        for c in g[u]:
            if c != parent[u]:
                up[c] = w(u) + max(up[u], down[u])               # SAI: down[u] có thể đi qua chính c
    return [max(down[u], up[u]) for u in range(n)]

print('sai :', wrong_top1_only(examples[0][0]), '| đúng:', Solution().timeTaken(examples[0][0]))

sai : [2, 4, 4] | đúng: [2, 4, 3]


**Vì sao sai:** từ đỉnh `1`, đi lên `0` rồi bản sai lại "đi xuống" về chính nhánh `1` (`down[0]` qua con `2` hoặc qua `1`) — phải loại nhánh của `c`.

### Lỗi 4 — DFS đệ quy trên cây sâu

In [16]:
def recursive_down(edges):
    g = build(edges)
    def dfs(u, p):
        best = 0
        for v in g[u]:
            if v != p:
                best = max(best, w(v) + dfs(v, u))
        return best
    return dfs(0, -1)

old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    recursive_down([[i, i + 1] for i in range(5000)])
    print('không lỗi')
except RecursionError:
    print('RecursionError với cây đường thẳng 5001 đỉnh (giới hạn mặc định) -> dùng duyệt lặp')
finally:
    sys.setrecursionlimit(old)

RecursionError với cây đường thẳng 5001 đỉnh (giới hạn mặc định) -> dùng duyệt lặp


**Vì sao sai:** cây `10^5` đỉnh dạng đường thẳng vượt giới hạn đệ quy mặc định của Python.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lan truyền trên cây, mỗi đỉnh có độ trễ riêng" ⇒ khoảng cách với trọng số đỉnh đích.
- "Đáp án cho **mọi** gốc" ⇒ rerooting với top-2.
- Khoảng cách không đối xứng ⇒ cẩn thận với các mẹo dựa trên đối xứng (đường kính).

## 15. Biến thể của bài

1. Trọng số đỉnh tùy ý `t[v] ≥ 0` ⇒ thay `w(v)`.
2. Trọng số cạnh (đối xứng) ⇒ mẹo đường kính dùng được (LC 3313 họ).
3. Hỏi **đỉnh** được đánh dấu cuối cùng (kèm chỉ số) ⇒ lưu đỉnh đạt max trong top-2.
4. Bắt đầu từ **nhiều** đỉnh cùng lúc ⇒ BFS/Dijkstra đa nguồn.

In [17]:
# Biến thể 1: trọng số đỉnh tùy ý
def general_weights(edges, t):
    g = build(edges); n = len(g)
    parent = [-1] * n; order = []; seen = [False] * n; seen[0] = True; st = [0]
    while st:
        u = st.pop(); order.append(u)
        for v in g[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; st.append(v)
    b1 = [0] * n; b2 = [0] * n; via = [-1] * n
    for u in reversed(order):
        for c in g[u]:
            if c != parent[u]:
                val = t[c] + b1[c]
                if val > b1[u]:
                    b2[u], b1[u], via[u] = b1[u], val, c
                elif val > b2[u]:
                    b2[u] = val
    up = [0] * n
    for u in order:
        for c in g[u]:
            if c != parent[u]:
                up[c] = t[u] + max(up[u], b2[u] if via[u] == c else b1[u])
    return [max(b1[u], up[u]) for u in range(n)]

rng = random.Random(15)
for _ in range(200):
    n = rng.randint(2, 12); e = random_tree(n); t = [rng.randint(0, 9) for _ in range(n)]
    g = build(e); exp = []
    for s in range(n):
        d = [-1] * n; d[s] = 0; st = [s]
        while st:
            u = st.pop()
            for v in g[u]:
                if d[v] < 0:
                    d[v] = d[u] + t[v]; st.append(v)
        exp.append(max(d))
    assert general_weights(e, t) == exp
assert general_weights(examples[2][0], [w(v) for v in range(5)]) == examples[2][1]
print('Biến thể trọng số đỉnh tùy ý khớp vét cạn (và trùng bài gốc khi t = w).')

Biến thể trọng số đỉnh tùy ý khớp vét cạn (và trùng bài gốc khi t = w).


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Duyệt từ mọi đỉnh | `O(n²)` | Oracle |
| 2 | Dijkstra từ mọi đỉnh | `O(n² log n)` | Tổng quát cho đồ thị |
| 3 | Rerooting top-2 | `O(n)` | **Nộp bài** |
| 4 | DP cạnh có hướng (memo) | `O(Σ deg²)` | Dễ viết, chậm với hình sao |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `edges = [[1, 0], [1, 2]]`.

**Bài 2 (tính tay):** đường thẳng `0 – 1 – 2 – 3`.

**Bài 3 (điền TODO):** trả về luôn đỉnh được đánh dấu **cuối cùng** cho mỗi `i`.

**Bài 4 (tự viết lại):** biến thể 4 — bắt đầu từ một tập nguồn `S`.

**Bài 5 (mini challenge):** tìm `i` có `times[i]` nhỏ nhất ("tâm" theo nghĩa này) và so với tâm thông thường.

In [18]:
def last_marked_todo(edges):
    # TODO
    raise NotImplementedError

def multi_source_todo(edges, S):
    # TODO
    raise NotImplementedError

def weighted_center_todo(edges):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Từ `1`: hai đỉnh chẵn kề ⇒ 2. (2) Từ `0`: `1` (lẻ) tại 1, `2` (chẵn) tại 3. (3) `[3, 2, 3]`.

**Bài 2.** (1) Từ `0`: `1 + 2 + 1 = 4`. (2) Từ `3`: `2 + 1 + 2 = 5`. (3) Từ `1`: xa nhất là `3` (`w(2) + w(3) = 3`); từ `2`: tới `0` mất `w(1) + w(0) = 3` ⇒ `[4, 3, 3, 5]`.

**Bài 3.** (1) Lưu kèm đỉnh đạt max trong `down` và `up`. (2) Top-2 cũng lưu đỉnh. (3) Đối chiếu vét cạn (có thể nhiều đáp án — so khoảng cách).

**Bài 4.** (1) Dijkstra đa nguồn với trọng số đỉnh đích. (2) Trên cây có thể dùng DP hai lượt. (3) Vét cạn.

**Bài 5.** (1) `argmin times`. (2) Tâm thông thường: `argmin` độ lệch tâm không trọng số. (3) Hai tâm có thể khác nhau.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 5
assert Solution().timeTaken([[1, 0], [1, 2]]) == [3, 2, 3]
line = Solution().timeTaken([[0, 1], [1, 2], [2, 3]])
assert line == brute([[0, 1], [1, 2], [2, 3]]) == [4, 3, 3, 5]

def weighted_center(edges):
    t = Solution().timeTaken(edges)
    g = build(edges); n = len(g)
    plain = []
    for s in range(n):
        d = [-1] * n; d[s] = 0; q = deque([s])
        while q:
            u = q.popleft()
            for v in g[u]:
                if d[v] < 0:
                    d[v] = d[u] + 1; q.append(v)
        plain.append(max(d))
    return t.index(min(t)), plain.index(min(plain))

print('Bài 1: [3, 2, 3] | Bài 2:', line, '| Bài 5 (tâm có trọng số, tâm thường) ví dụ 3:', weighted_center(examples[2][0]))

Bài 1: [3, 2, 3] | Bài 2: [4, 3, 3, 5] | Bài 5 (tâm có trọng số, tâm thường) ví dụ 3: (2, 0)


In [20]:
# Bài 3, 4
def last_marked(edges):
    g = build(edges)
    res = []
    for s in range(len(g)):
        d = dist_from(g, s); m = max(d); res.append(d.index(m))
    return res

def multi_source(edges, S):
    g = build(edges); n = len(g)
    d = [float('inf')] * n; pq = []
    for s in S:
        d[s] = 0; pq.append((0, s))
    heapq.heapify(pq)
    while pq:
        du, u = heapq.heappop(pq)
        if du > d[u]:
            continue
        for v in g[u]:
            if du + w(v) < d[v]:
                d[v] = du + w(v); heapq.heappush(pq, (d[v], v))
    return max(d)

rng = random.Random(4)
for _ in range(200):
    n = rng.randint(2, 12); e = random_tree(n)
    lm = last_marked(e); t = Solution().timeTaken(e); g = build(e)
    assert all(dist_from(g, s)[lm[s]] == t[s] for s in range(n))
    S = rng.sample(range(n), rng.randint(1, min(3, n)))
    assert multi_source(e, S) == max(min(dist_from(g, s)[v] for s in S) for v in range(n))
print('Bài 3, 4 khớp vét cạn.')

Bài 3, 4 khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
w(v) = 1 nếu v lẻ, 2 nếu chẵn         # trọng số của đỉnh ĐƯỢC đánh dấu
hậu thứ tự: val = w(c) + down[c] -> cập nhật top-2 (b1, b2, via) của cha
tiền thứ tự: up[c] = w(u) + max(up[u], b2[u] nếu via[u] == c else b1[u])
times[u] = max(b1[u], up[u])
```

**Checklist:**

- [ ] Trọng số theo đỉnh **đích**?
- [ ] Không dùng mẹo đường kính (không đối xứng)?
- [ ] Top-2 để loại nhánh của chính con?
- [ ] Duyệt lặp (cây sâu)?
- [ ] `up` của gốc = 0?